In [1]:
!pip install findspark

In [ ]:
#pyspark --packages com.datastax.spark:spark-cassandra-connector_2.12:3.1.0
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession
import os
import time
import datetime
import pyspark.sql.functions as sf
from uuid import *
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.functions import when
from pyspark.sql.functions import col
from pyspark.sql.types import *
from pyspark.sql.functions import lit
from pyspark import SparkConf, SparkContext
from uuid import * 
from uuid import UUID
import time_uuid 
from pyspark.sql import Row
from pyspark.sql.functions import udf
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.window import Window as W

spark = SparkSession.builder \
    .appName("CDC_Final_Test") \
    .config('spark.jars.packages', 
            'com.datastax.spark:spark-cassandra-connector_2.12:3.4.1,' + 
            'mysql:mysql-connector-java:8.0.28') \
    .config('spark.cassandra.connection.host', '127.0.0.1') \
    .config('spark.cassandra.auth.username', 'cassandra') \
    .config('spark.cassandra.auth.password', 'cassandra') \
    .getOrCreate()

print("--- [OK] Spark đã sẵn sàng với cả Driver Cassandra và MySQL ---")
def get_latest_time_cassandra():
    data = spark.read.format("org.apache.spark.sql.cassandra").options(table = 'tracking',keyspace = 'keyspace_name').load()
    cassandra_latest_time = data.agg({'ts':'max'}).take(1)[0][0]
    return cassandra_latest_time

def get_mysql_latest_time(url, driver, user, password):    
    sql = """(select max(updated_at) from events) data"""
    mysql_time_df = spark.read.format('jdbc').options(url=url, driver=driver, dbtable=sql, user=user, password=password).load()
    
    # Lấy giá trị thô ra
    raw_time = mysql_time_df.take(1)[0][0]
    
    if raw_time is None:
        return '1998-01-01 23:59:59'
    
    # Dùng pandas để ép kiểu an toàn bất kể raw_time là str hay datetime
    mysql_latest = pd.to_datetime(raw_time).strftime('%Y-%m-%d %H:%M:%S')
    return mysql_latest

def calculating_clicks(df):
    clicks_data = df.filter(df.custom_track == 'click')
    clicks_data = clicks_data.na.fill({'bid':0})
    clicks_data = clicks_data.na.fill({'job_id':0})
    clicks_data = clicks_data.na.fill({'publisher_id':0})
    clicks_data = clicks_data.na.fill({'group_id':0})
    clicks_data = clicks_data.na.fill({'campaign_id':0})
    clicks_data.registerTempTable('clicks')
    clicks_output = spark.sql("""select job_id , date(ts) as date , hour(ts) as hour , publisher_id , campaign_id , group_id , avg(bid) as bid_set, count(*) as clicks , sum(bid) as spend_hour from clicks
    group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id """)
    return clicks_output 
    
def calculating_conversion(df):
    conversion_data = df.filter(df.custom_track == 'conversion')
    conversion_data = conversion_data.na.fill({'job_id':0})
    conversion_data = conversion_data.na.fill({'publisher_id':0})
    conversion_data = conversion_data.na.fill({'group_id':0})
    conversion_data = conversion_data.na.fill({'campaign_id':0})
    conversion_data.registerTempTable('conversion')
    conversion_output = spark.sql("""select job_id , date(ts) as date , hour(ts) as hour , publisher_id , campaign_id , group_id , count(*) as conversions  from conversion
    group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id """)
    return conversion_output 
    
def calculating_qualified(df):    
    qualified_data = df.filter(df.custom_track == 'qualified')
    qualified_data = qualified_data.na.fill({'job_id':0})
    qualified_data = qualified_data.na.fill({'publisher_id':0})
    qualified_data = qualified_data.na.fill({'group_id':0})
    qualified_data = qualified_data.na.fill({'campaign_id':0})
    qualified_data.registerTempTable('qualified')
    qualified_output = spark.sql("""select job_id , date(ts) as date , hour(ts) as hour , publisher_id , campaign_id , group_id , count(*) as qualified  from qualified
    group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id """)
    return qualified_output
    
def calculating_unqualified(df):
    unqualified_data = df.filter(df.custom_track == 'unqualified')
    unqualified_data = unqualified_data.na.fill({'job_id':0})
    unqualified_data = unqualified_data.na.fill({'publisher_id':0})
    unqualified_data = unqualified_data.na.fill({'group_id':0})
    unqualified_data = unqualified_data.na.fill({'campaign_id':0})
    unqualified_data.registerTempTable('unqualified')
    unqualified_output = spark.sql("""select job_id , date(ts) as date , hour(ts) as hour , publisher_id , campaign_id , group_id , count(*) as unqualified  from unqualified
    group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id """)
    return unqualified_output
    
def process_final_data(clicks_output,conversion_output,qualified_output,unqualified_output):
    final_data = clicks_output.join(conversion_output,['job_id','date','hour','publisher_id','campaign_id','group_id'],'full').\
    join(qualified_output,['job_id','date','hour','publisher_id','campaign_id','group_id'],'full').\
    join(unqualified_output,['job_id','date','hour','publisher_id','campaign_id','group_id'],'full')
    return final_data 
    
def process_cassandra_data(df):
    clicks_output = calculating_clicks(df)
    conversion_output = calculating_conversion(df)
    qualified_output = calculating_qualified(df)
    unqualified_output = calculating_unqualified(df)
    final_data = process_final_data(clicks_output,conversion_output,qualified_output,unqualified_output)
    return final_data

def retrieve_company_data(url,driver,user,password):
    sql = """(SELECT id as job_id, company_id, group_id, campaign_id FROM job) test"""
    company = spark.read.format('jdbc').options(url=url, driver=driver, dbtable=sql, user=user, password=password).load()
    return company 
    
def import_to_mysql(output):
    final_output = output.select('job_id','date','hour','publisher_id','company_id','campaign_id','group_id','unqualified','qualified','conversions','clicks','bid_set','spend_hour')
    final_output = final_output.withColumnRenamed('date','dates').withColumnRenamed('hour','hours').withColumnRenamed('qualified','qualified_application').\
    withColumnRenamed('unqualified','disqualified_application').withColumnRenamed('conversions','conversion')
    final_output = final_output.withColumn('sources',lit('Cassandra'))
    final_output.printSchema()
    final_output = final_output.withColumn('updated_at', sf.current_timestamp())
    final_output.write.format("jdbc") \
    .option("driver","com.mysql.cj.jdbc.Driver") \
    .option("url", "jdbc:mysql://localhost:3306/recruitment_dw") \
    .option("dbtable", "events") \
    .mode("append") \
    .option("user", "root") \
    .option("password", "root") \
    .save()
    return print('Data imported successfully')

def main_task(mysql_time):
    host = 'localhost'
    port = '3306'
    db_name = 'schema_name'
    user = 'root'
    password = 'root'
    url = 'jdbc:mysql://' + host + ':' + port + '/' + db_name
    driver = "com.mysql.cj.jdbc.Driver"
    print('The host is ' ,host)
    print('The port using is ',port)
    print('The db using is ',db_name)
    print('-----------------------------')
    print('Retrieving data from Cassandra')
    print('-----------------------------')
    df = spark.read.format("org.apache.spark.sql.cassandra").options(table="tracking",keyspace="keyspace_name").load().where(col('ts')>= mysql_time)
    print('-----------------------------')
    print('Selecting data from Cassandra')
    print('-----------------------------')
    df = df.select('ts','job_id','custom_track','bid','campaign_id','group_id','publisher_id')
    df = df.filter(df.job_id.isNotNull())
    df.printSchema()
#   process_df = process_df(df)
    print('-----------------------------')
    print('Processing Cassandra Output')
    print('-----------------------------')
    cassandra_output = process_cassandra_data(df)
    print('-----------------------------')
    print('Merge Company Data')
    print('-----------------------------')
    company = retrieve_company_data(url,driver,user,password)
    print('-----------------------------')
    print('Finalizing Output')
    print('-----------------------------')
    final_output = cassandra_output.join(company,'job_id','left').drop(company.group_id).drop(company.campaign_id)
    print('-----------------------------')
    print('Import Output to MySQL')
    print('-----------------------------')
    import_to_mysql(final_output)
    return print('Task Finished')

host = 'localhost'
port = '3306'
db_name = 'recruitment_dw '
user = 'root'
password = 'root'
url = 'jdbc:mysql://' + host + ':' + port + '/' + db_name
driver = "com.mysql.cj.jdbc.Driver"
while True :
    start_time = datetime.datetime.now()
    cassandra_time = get_latest_time_cassandra()
    print('Cassandra latest time is {}'.format(cassandra_time))
    mysql_time = get_mysql_latest_time(url,driver,user,password)
    print('MySQL latest time is {}'.format(mysql_time))
    if cassandra_time > mysql_time : 
        print("Do main task")
        main_task(mysql_time)
    else :
        print("No new data found")
    end_time = datetime.datetime.now()
    execution_time = (end_time - start_time).total_seconds()
    print('Job takes {} seconds to execute'.format(execution_time))
    time.sleep(10)


your 131072x1 screen size is bogus. expect trouble
26/03/30 14:52:28 WARN Utils: Your hostname, shiba resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/30 14:52:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/shibe/python_envs/cdc_project/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/shibe/.ivy2/cache
The jars for the packages stored in: /home/shibe/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
mysql#mysql-connector-java added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0f543c9f-e718-46d0-86fb-cbb6e1a93df3;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.4.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.4.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	fou

--- [OK] Spark đã sẵn sàng với cả Driver Cassandra và MySQL ---


26/03/30 14:52:33 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because the driver expects authentication
26/03/30 14:52:34 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because the driver expects authentication
26/03/30 14:52:34 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because the driver expects authentication
26/03/30 14:52:34 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because the driver expects authentication
26/03/30 14:52:34 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because the driver expects authentication
26/03/30 14:52:34 WARN PlainTextAuthProviderBase: [] /127.0.0.1:9042 did not send an authentication challenge; This is suspicious because th

...

Cassandra latest time is 2026-03-30 14:54:09
MySQL latest time is 2026-03-30 14:53:57
Do main task
The host is  localhost
The port using is  3306
The db using is  schema_name
-----------------------------
Retrieving data from Cassandra
-----------------------------
-----------------------------
Selecting data from Cassandra
-----------------------------
root
 |-- ts: string (nullable = true)
 |-- job_id: integer (nullable = true)
 |-- custom_track: string (nullable = true)
 |-- bid: double (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- group_id: integer (nullable = true)
 |-- publisher_id: integer (nullable = true)

-----------------------------
Processing Cassandra Output
-----------------------------
-----------------------------
Merge Company Data
-----------------------------
-----------------------------
Finalizing Output
-----------------------------
-----------------------------
Import Output to MySQL
-----------------------------
root
 |-- job_id: integer (n

26/03/30 14:54:20 WARN CqlRequestHandler: Query '[2 values] SELECT "bid", "campaign_id", "custom_track", "group_id", "job_id", "publisher_id", "ts" FROM keyspace_name.tracking WHERE token("create_time") > ? AND token("create_time") <= ?   ALLOW FILTERING ["partition key token"=-8292553682435006495, "partition key token"=-7914405992246235402]' generated server side warning(s): Read 178 live rows and 1583 tombstone cells for query SELECT bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts FROM keyspace_name.tracking WHERE token(create_time) > -8292553682435006495 AND token(create_time) <= -7914405992246235402 LIMIT 1000; token -7916883094938723450 (see tombstone_warn_threshold)
26/03/30 14:54:20 WARN CqlRequestHandler: Query '[2 values] SELECT "bid", "campaign_id", "custom_track", "group_id", "job_id", "publisher_id", "ts" FROM keyspace_name.tracking WHERE token("create_time") > ? AND token("create_time") <= ?   ALLOW FILTERING ["partition key token"=-1280091385655497248, 

Data imported successfully
Task Finished
Job takes 1.703521 seconds to execute


KeyboardInterrupt: 

26/03/30 15:10:02 WARN ChannelPool: [s0|/127.0.0.1:9042]  Error while opening new channel (ConnectionInitException: [s0|id: 0x9ceea141, L:/127.0.0.1:50818 - R:localhost/127.0.0.1:9042] Protocol initialization request, step 1 (STARTUP {CQL_VERSION=3.0.0, DRIVER_NAME=DataStax Java driver for Apache Cassandra(R), DRIVER_VERSION=4.13.0, CLIENT_ID=a2c2cc29-5b56-4074-b405-a7d79373b340, APPLICATION_NAME=Spark-Cassandra-Connector-local-1774882351049}): unexpected failure (com.datastax.oss.driver.api.core.connection.ClosedConnectionException: Unexpected error on channel))
26/03/30 15:10:02 WARN ControlConnection: [s0] Error connecting to Node(endPoint=localhost/127.0.0.1:9042, hostId=bb8894e3-05b3-4196-86b9-640e25df1c74, hashCode=274f7e5d), trying next node (ConnectionInitException: [s0|control|id: 0xcbfe956a, L:/127.0.0.1:50918 - R:localhost/127.0.0.1:9042] Protocol initialization request, step 1 (STARTUP {CQL_VERSION=3.0.0, DRIVER_NAME=DataStax Java driver for Apache Cassandra(R), DRIVER_VERS